# From movie ratings to live trends

**Data Stream · 90-minute practical · Student notebook**

We have historical movie ratings from a Netflix-like service. First we explore
what people rated, then estimate what they might like, and finally monitor what
people are playing **right now**.

```text
MovieLens CSV → DataFrames → ALS recommendations
                                 ↓ a new question
synthetic live plays → Structured Streaming → trending movies
```

**How to work:** run the **Given** cells unchanged. Complete the **TODO** edits
and questions in order. Starter cells are valid Python, but some use provisional
values or incomplete analyses; running them is not the same as solving them.
All required API patterns are shown locally. Optional tasks can be skipped.

Budget: check 5 min · DataFrames 15 min · motivation 5 min · ALS 25 min ·
transition 5 min · streaming 25 min · buffer/discussion 10 min.


## Part I — Historical data with DataFrames (20 min including check)

**Given — environment check.** You should already have run
`python check_environment.py` in your activated Miniforge environment and seen
**Environment ready.** If not, do that before starting the kernel.

A `SparkSession` is our entry point. A Spark DataFrame is a table whose partitions
can be processed across workers. Here everything stays on one laptop, with two
worker threads. The same DataFrame API also applies on a cluster.

The helper below selects this environment's Java/Python without requiring shell
variables. The small partition count avoids launching hundreds of tiny tasks.


In [ ]:
from pathlib import Path
import os
import sys

# Works when opened from the lab directory or one of its subdirectories.
ROOT = Path.cwd().resolve()
if not (ROOT / "environment.yml").exists():
    ROOT = ROOT.parent
assert (ROOT / "environment.yml").exists(), "Launch Jupyter from the lab directory."
sys.path.insert(0, str(ROOT))
from lab_support import configure_local_environment
configure_local_environment()

from pyspark.sql import SparkSession, functions as F, types as T
spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("data-stream-lab")
    .config("spark.driver.memory", "768m")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.default.parallelism", "2")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Python:", sys.executable)
print("Spark:", spark.version, "| master:", spark.sparkContext.master)


**Given — explicit schemas.** IDs are integers, ratings are floating-point values
on a 0.5–5 scale, and the historical timestamp is Unix seconds (UTC). Explicit
schemas avoid a scan to guess types. CSV uses doubled quotes in movie titles.
The included data are MovieLens small (2018); see `data/MOVIELENS_README.txt`
for its original terms and attribution.


In [ ]:
movies_schema = "movieId INT, title STRING, genres STRING"
ratings_schema = "userId INT, movieId INT, rating FLOAT, timestamp LONG"
movies = (spark.read.option("header", True).option("escape", '"')
          .schema(movies_schema).csv((ROOT / "data/movies.csv").as_uri()))
ratings = (spark.read.option("header", True).schema(ratings_schema)
           .csv((ROOT / "data/ratings.csv").as_uri()))


### Exercise 1 — Inspect the data (3 min)

**TODO:** display the first five ratings, print their schema, count all ratings,
distinct users, and distinct **rated** movies. Also count the metadata rows.
Useful patterns: `df.show(5)`, `df.printSchema()`, `df.count()`,
`df.select("column").distinct().count()`.

Why might the movie metadata count differ from the number of rated movies?


In [ ]:
# TODO: add the schema and three remaining counts using the patterns above.
ratings.show(5)
print("Ratings:", ratings.count())
print("Users: ??")
print("Rated movies: ??")
print("Movie metadata rows: ??")


**TODO — observation:** write your counts and explanation here.


### Exercise 2 — Aggregate and join (4 min)

**TODO:** extend the starting aggregation to compute `avg_rating` using
`F.avg("rating").alias("avg_rating")`. Keep `n_ratings`. Join metadata using
`.join(movies, "movieId")`, then display `title`, `n_ratings`, and `avg_rating`.

`groupBy` defines groups; `agg` computes one result per group; a join combines
tables on their shared key. `alias` gives a column a readable name.


In [ ]:
# TODO: add average rating to agg, then join movies and select readable columns.
movie_summary = ratings.groupBy("movieId").agg(
    F.count("*").alias("n_ratings"),
)
movie_summary.show(5)


### Exercise 3 — Best-rated movies (4 min)

**TODO:** find the **ten** movies with the highest mean rating among those with
**at least 100 ratings**. Change the two provisional values below.
The full aggregation is supplied again so this exercise can run independently
of your Exercise 2 edits. After solving Exercise 2, you can reuse `movie_summary`.

Predict why a minimum rating count matters. A movie with one 5-star rating can
otherwise outrank a movie rated by hundreds of people.


In [ ]:
# TODO: replace these provisional values.
minimum_ratings = 1
top_n = 5

ranking_source = ratings.groupBy("movieId").agg(
    F.count("*").alias("n_ratings"),
    F.avg("rating").alias("avg_rating"),
).join(movies, "movieId")
best_movies = (ranking_source.filter(F.col("n_ratings") >= minimum_ratings)
               .orderBy(F.desc("avg_rating"), F.desc("n_ratings"), "movieId")
               .limit(top_n))
best_movies.select("title", "n_ratings", "avg_rating").show(truncate=False)


**TODO — observation:** what changed when you required 100 ratings?


### Exercise 4 — Lazy execution (4 min)

**Given:** `filter`, `select`, `groupBy`, and `join` are transformations: they
describe a computation. Actions such as `show`, `count`, and `collect` ask Spark
to produce results. `collect` brings *all* rows to the driver; use it only for
small results. `explain` prints the plan, rather than computing the result rows.

**TODO:** run the cell. Which line asks for actual result rows? Locate a scan,
filter and aggregate in the plan. An `Exchange` is data redistribution between
partitions; its exact placement can vary with Spark's optimizer.


In [ ]:
pipeline = ratings.filter(F.col("rating") >= 4).groupBy("movieId").count()
pipeline.explain()
print(pipeline)
pipeline.show(5)


**TODO — answer:** when does this pipeline run?


## Why Spark for machine learning? (5 min)

Spark originated at UC Berkeley to support workloads that MapReduce handled
inefficiently, including iterative machine learning and interactive analysis.
Many algorithms revisit the same working data over several iterations.

Traditional MapReduce workflows materialized intermediate results between jobs.
Spark made it possible to retain reusable working sets in memory, while still
supporting distributed computation and recovery. Caching is useful when data
are reused; it is not automatic for every DataFrame and memory is finite.

Matrix-factorization recommenders are a good example of such iterative work.
The Netflix Prize popularized this kind of recommendation problem; it did not
cause Spark's creation. See the [original Berkeley Spark paper](https://www2.eecs.berkeley.edu/Pubs/TechRpts/2010/EECS-2010-53.pdf).


## Part II — A miniature Netflix recommendation problem (25 min)

Our sparse ratings matrix has many missing entries:

```text
             movie 1   movie 2   movie 3
user 1           5         ?         2
user 2           ?         4         5
user 3           1         ?         4
```

We approximate $R \approx X Y^\top$, or $\hat r_{u,i} = x_u^\top y_i$.
Each user and each movie has a short **latent vector**. Their dot product
estimates a rating. The coordinates are learned; they do not come with labels
such as 'comedy preference'. Missing entries are unknown, not zero ratings.

**Alternating Least Squares (ALS)** holds movie factors fixed to estimate user
factors, then holds user factors fixed to estimate movie factors, repeatedly.
`rank` is the vector length, `maxIter` limits iterations, and `regParam` controls
regularization. We use explicit ratings (`implicitPrefs=False`, the default).
We will not derive the optimization equations.

### Exercise 5 — Train/test split (4 min)

**TODO:** change the provisional split below to **80% train / 20% test**.
Keep the seed fixed. `randomSplit([0.8, 0.2], seed=42)` gives approximate sizes,
not exact proportions. We fit on train and evaluate on held-out test ratings.
This teaching split is random; a production forecast would need a temporal
evaluation design.


In [ ]:
# TODO: use an 80/20 split.
train, test = ratings.randomSplit([0.7, 0.3], seed=42)

train = train.cache()
test = test.cache()
train_count, test_count = train.count(), test.count()
print("Train:", train_count, "Test:", test_count)


### Exercise 6 — Fit ALS (8 min)

**TODO:** set `rank=10` and `maxIter=5` in the provided estimator, then fit it.
The initial values keep the starter runnable. `fit(train)` returns a model;
`model.transform(test)` will add predictions. Two user/item blocks keep this
small problem inexpensive on two worker threads. The first fit may take
tens of seconds on a slower laptop; later actions are generally shorter.

Some test users or movies may never occur in training. ALS then has no latent
vector for them. `coldStartStrategy="drop"` excludes such rows from predictions
so RMSE is finite. We must also report how many test rows remain; dropping them
does not solve the cold-start problem.


In [ ]:
# TODO: change rank and maxIter.

from pyspark.ml.recommendation import ALS
als = ALS(
    userCol="userId", itemCol="movieId", ratingCol="rating",
    rank=2, maxIter=1, regParam=0.1,
    coldStartStrategy="drop", seed=42,
    numUserBlocks=2, numItemBlocks=2,
)
model = als.fit(train)


### Exercise 7 — Evaluate (6 min)

**TODO:** replace the starter metric `"mae"` with `"rmse"`, then inspect
`userId`, `movieId`, `rating`, `prediction`. RMSE is the square root of the mean
squared error, measured on the rating scale. Lower is better on the same test
set. `RegressionEvaluator` takes the observed and predicted column names.

The given coverage calculation tells us what fraction of the test set the
model can predict. Predictions are not clipped to the 0.5–5 rating range.


In [ ]:
# TODO: evaluate RMSE.

from pyspark.ml.evaluation import RegressionEvaluator
predictions = model.transform(test).cache()
predicted_count = predictions.count()
coverage = predicted_count / test_count
evaluator = RegressionEvaluator(
    metricName="mae", labelCol="rating", predictionCol="prediction"
)
score = evaluator.evaluate(predictions)
print(f"{evaluator.getMetricName().upper()}: {score:.3f}")
print(f"Coverage: {predicted_count}/{test_count} ({coverage:.1%})")
predictions.select("userId", "movieId", "rating", "prediction").show(8)


**TODO — interpretation:** can RMSE alone tell us whether recommendations are useful?


### Exercise 8 — Show recommendations with titles (7 min)

**TODO:** request **five** recommendations for user 42, using
`model.recommendForUserSubset(selected_users, 5)`. The provided
`posexplode` turns each recommendation array into rows and preserves its position
(starting at zero). Complete the display by changing `movieId` to `title` in the
last `select`. The metadata join is supplied.

The scores estimate preference, not probabilities. This basic Spark method may
recommend movies the user has already rated; filtering seen items is an optional
extension and would require requesting more candidates before filtering.


In [ ]:
# TODO: request five and display titles.

selected_users = train.filter(F.col("userId") == 42).select("userId").distinct()
recommendations = model.recommendForUserSubset(selected_users, 3)
recommendation_rows = recommendations.select(
    "userId", F.posexplode("recommendations").alias("position", "rec")
).select(
    "userId", (F.col("position") + 1).alias("rank"),
    F.col("rec.movieId").alias("movieId"), F.col("rec.rating").alias("score"),
)
titled_recommendations = recommendation_rows.join(movies, "movieId")
titled_recommendations.orderBy("userId", "rank").select(
    "userId", "rank", "movieId", "score"
).show(truncate=False)


**Optional — rank comparison (buffer time):** after recording RMSE/coverage,
repeat the fit with rank 5 and the same split, iterations and seed. Do not tune
on the test set repeatedly; a real study would reserve a validation set.
Discuss whether a lower RMSE guarantees a better top-five list.

**Given — release caches:** we no longer need these reused batch tables.


In [ ]:
predictions.unpersist()
train.unpersist()
test.unpersist()


## A new question: what is happening right now? (5 min)

Historical ratings help answer **“What will this user like?”** A streaming
platform also receives events continuously: **“What is happening right now?”**

```text
timestamp (UTC)       userId  movieId  eventType
12:01:02                 12       42  play
12:01:05                 91       17  play
12:01:08                 12       42  play

batch ML:          historical ratings → model training → recommendations
stream processing: continuous events → online aggregation → current trends
```

We simulate plays, not real MovieLens viewing histories. We **do not retrain ALS
continuously**. Both branches use DataFrames, but solve different questions.

## Part III — Structured Streaming (25 min)

**Given — incoming files.** A streaming DataFrame represents an unbounded table.
New files add rows. Transformations look like batch transformations, but a
`writeStream` query incrementally processes new data. You cannot call `show()`
directly on a streaming DataFrame: inspect its output sink instead.

In a second terminal in the repository directory, run:

```bash
conda activate data-stream-spark
python streaming/generate_events.py
```

Leave it running through Exercise 11. The default producer emits 40 plays every
2 seconds, across 20 movies and multiple users, with uneven baseline popularity.
After 20 seconds a movie gets a burst. The terminal names it. **Ctrl+C** stops
the producer cleanly. You can inspect a JSON file in Jupyter's file browser.

The next cell adds a small finite replay if the input directory is empty, so the
notebook also runs without a live producer. Query checkpoints track progress and
state. Each demonstration uses a fresh checkpoint and replays available files;
these memory tables are for learning and do not support production recovery.


In [ ]:
import tempfile
import time
from streaming.generate_events import seed_demo

# Automated tests may redirect this; normally the documented directory is used.
STREAM_ROOT = Path(os.environ.get("SPARK_LAB_STREAM_ROOT", ROOT / "streaming/runtime")).resolve()
incoming = STREAM_ROOT / "events"
incoming.mkdir(parents=True, exist_ok=True)
if not any(incoming.glob("events-*.json")):
    seed_demo(incoming)
checkpoints = STREAM_ROOT / "checkpoints"
checkpoints.mkdir(exist_ok=True)

def fresh_checkpoint():
    return Path(tempfile.mkdtemp(prefix="query-", dir=checkpoints)).as_uri()

def stop_query(name):
    """Stop an older run of a query, so rerunning its start cell does not clash."""
    for query in spark.streams.active:
        if query.name == name:
            query.stop()

def show_progress(query):
    """Print Spark's record of the query's most recent micro-batch."""
    deadline = time.monotonic() + 30
    while query.lastProgress is None and query.isActive and time.monotonic() < deadline:
        time.sleep(0.5)  # a new query needs a moment for its first batch
    progress = query.lastProgress
    if progress is None:
        print(f"{query.name}: no micro-batch has completed yet")
    else:
        print(f"{query.name}: micro-batch {progress['batchId']} read "
              f"{progress['numInputRows']} new rows")

def warn_if_no_producer(max_age=10):
    newest = max((f.stat().st_mtime for f in incoming.glob("events-*.json")), default=0)
    if time.time() - newest > max_age:
        print(f"No new event file in {max_age} seconds, so nothing new is streaming in. "
              "Start the producer in a second terminal (see above).")

print("Reading files from:", incoming)


### Exercise 9 — Read and inspect the stream (5 min)

**TODO:** use the given schema and reader; add `print(events.isStreaming)` and
`events.printSchema()`. Predict the value of `isStreaming`.

Then run the supplied memory-sink cell. `availableNow=True` processes the files
available when that query starts and then terminates. Its output remains in a
small in-memory SQL table. This bounded inspection reads a snapshot, as a
batch job would. From Exercise 10 on, queries keep running until you stop them.


In [ ]:
event_schema = T.StructType([
    T.StructField("timestamp", T.TimestampType()),
    T.StructField("userId", T.IntegerType()),
    T.StructField("movieId", T.IntegerType()),
    T.StructField("eventType", T.StringType()),
])
events = (spark.readStream.schema(event_schema).json(incoming.as_uri())
          .filter(F.col("eventType") == "play"))

# TODO: print isStreaming and inspect the schema.


In [ ]:
raw_query = (events.writeStream.format("memory").queryName("incoming_plays")
             .outputMode("append").option("checkpointLocation", fresh_checkpoint())
             .trigger(availableNow=True).start())
try:
    if not raw_query.awaitTermination(60):
        raise TimeoutError("Reading the replay took over 60 seconds; see query status.")
    spark.table("incoming_plays").orderBy(F.desc("timestamp")).show(5, truncate=False)
finally:
    raw_query.stop()


### Exercise 10 — Count views (6 min)

**TODO:** the starter groups by `userId`. Change it to `movieId` to answer
**“How many views does each movie receive?”** Then run the supplied start cell.

This query does not stop by itself. Every 5 seconds (its **trigger**), Spark
runs a **micro-batch**: it reads only the files that arrived since the previous
batch and adds them to the counts. `complete` mode replaces the memory table
with all current counts after each batch. Spark maintains **state**: the running
count for each key across batches.

Why can a global aggregation on a truly infinite stream never be “finished”?
Does a cumulative count necessarily identify what is popular *now*?


In [ ]:
# TODO: count per movie instead of per user.
view_counts = events.groupBy("userId").count()


In [ ]:
stop_query("movie_views")
count_query = (view_counts.writeStream.format("memory").queryName("movie_views")
               .outputMode("complete").option("checkpointLocation", fresh_checkpoint())
               .trigger(processingTime="5 seconds").start())
print("Started; the query keeps running in the background.")


**Given — peek at the running query.** Run this cell, wait about 10 seconds,
then run it again, two or three times. No cell is recomputing the counts: the
query updates them in the background. Watch the micro-batch number and the
counts grow. Leave the query running; Exercise 11 compares against it.


In [ ]:
show_progress(count_query)
spark.table("movie_views").orderBy(F.desc("count")).show(5)
warn_if_no_producer()


**TODO — answer:** explain why the counts need state and do not become final.


### Exercise 11 — Windowed trends (10 min)

**TODO:** compute views in **30-second event-time windows starting every 5
seconds**, using `F.window("timestamp", "30 seconds", "5 seconds")`.
Change the provisional slide below from 30 to 5 seconds, then run the given
query and display cells. The static metadata join supplies movie titles.

The window duration and slide describe **event time**. The trigger separately
asks Spark to check for data every **5 seconds of processing time**; slower
batches can delay updates. Windows are half-open `[start, end)`. A play belongs
to six overlapping windows with a 30-second duration and 5-second slide.

This short demonstration retains counts for all windows (`complete` mode).
Display only a recent **ended** window: otherwise a future-ending, partially
filled window can be mistaken for “the previous 30 seconds”. The optional
watermark section explains how a long-running service can bound window state.


In [ ]:
# TODO: use a 5-second slide.

windowed_counts = events.groupBy(
    F.window("timestamp", "30 seconds", "30 seconds"), "movieId"
).count()


In [ ]:
stop_query("windowed_views")
trend_query = (windowed_counts.writeStream.format("memory").queryName("windowed_views")
               .outputMode("complete").option("checkpointLocation", fresh_checkpoint())
               .trigger(processingTime="5 seconds").start())


**Given — watch for about a minute.** Every 5 seconds the display below
refreshes in place. It shows the latest *ended* 30-second window, then the
running totals from Exercise 10. Both queries keep updating in the background.
The display joins the static movie metadata to each memory-table snapshot.
Windows can still be revised by late arrivals; an empty window produces no row
rather than a row of zeros.

Rerun the cell to keep watching; interrupting it (■) stops only the display,
not the queries. **TODO:** identify the movie that rises during the producer's
burst. Does it also lead the running totals? Without the live producer, the
display shows the same replayed window each time.


In [ ]:
from IPython.display import clear_output

watch_seconds = 60
observed_trends = False
for observation in range(watch_seconds // 5):
    time.sleep(5)
    for query in (trend_query, count_query):
        if query.exception() is not None:
            raise query.exception()
    if not trend_query.isActive:
        raise RuntimeError("The trends query is not running; rerun its start cell.")
    snapshot = spark.table("windowed_views")
    ended = snapshot.filter(F.col("window.end") <= F.current_timestamp())
    # Format in Spark's UTC session, avoiding Python's local naive timestamps.
    latest_end = ended.agg(
        F.date_format(F.max("window.end"), "yyyy-MM-dd HH:mm:ss").alias("end")
    ).first()["end"]
    clear_output(wait=True)
    print(f"Refresh {observation + 1} of {watch_seconds // 5}")
    show_progress(trend_query)
    if latest_end is None:
        print("Waiting for an ended window with data...")
        continue
    current_trends = (ended.filter(F.col("window.end") == F.to_timestamp(F.lit(latest_end)))
                      .join(movies.select("movieId", "title"), "movieId"))
    print("\nLatest 30-second window, ending", latest_end, "UTC")
    current_trends.orderBy(F.desc("count"), "movieId").select(
        "window.start", "window.end", "title", "count"
    ).show(5, truncate=False)
    print("Running totals since the Exercise 10 query started")
    totals = spark.table("movie_views")
    if "movieId" in totals.columns:
        totals = totals.join(movies.select("movieId", "title"), "movieId").select("title", "count")
    else:
        print("(Titles appear once Exercise 10 counts per movieId.)")
    totals.orderBy(F.desc("count")).show(5, truncate=False)
    warn_if_no_producer()
    observed_trends = True


**Given — stop the queries** once you have finished watching. Until then, they
keep running a micro-batch every 5 seconds, even while no cell is running.


In [ ]:
trend_query.stop()
count_query.stop()
print("Active queries:", [query.name for query in spark.streams.active])


### Exercise 12 — Batch versus stream (4 min)

**TODO:** why is this different from computing the most popular movie in
`ratings.csv`? Mention finiteness, changing results, time windows and state.
Also distinguish the meaning of a rating from that of a synthetic play.


## Optional — late events and watermarks (buffer time)

Event occurrence time and arrival time can differ: an offline device may send a
play later. `events.withWatermark("timestamp", "30 seconds")` declares a
lateness threshold relative to the largest event timestamp processed. It is
not a timer that advances just because the wall clock moves.

With window aggregation and **append** mode, Spark emits a window after the
watermark passes its end and can remove old window state. Results are delayed;
excessively late data may be dropped. The threshold provides a guarantee for
data within the allowed lateness, not an exact drop boundary for every event.
Adding a watermark to our **complete**-mode sink would not bound its state,
because complete mode must keep the entire result table.

**Given — construct the optional plan only; no additional query is started.**
To try it later, use a new checkpoint and an append-mode sink, and send events
far enough into the future for event time and the watermark to advance.


In [ ]:
late_tolerant_counts = (events.withWatermark("timestamp", "30 seconds")
                       .groupBy(F.window("timestamp", "30 seconds", "5 seconds"), "movieId")
                       .count())


## Three roles, one DataFrame abstraction

```text
Data processing:       MovieLens CSV → transformations → aggregations / joins
Machine learning:      historical ratings → ALS → recommendation model
Stream processing:     live events → Structured Streaming → windows → trends

                    historical ratings
                           │
                           ▼
                       Spark ML
                           │
                           ▼
                       ALS model
                           │
                           ▼
                    recommendations

live user events
      │
      ▼
Structured Streaming
      │
      ▼
windowed aggregation
      │
      ▼
trending movies
```

A real platform could retrain its recommender periodically while streaming jobs
continuously monitor live activity. Spark supplies a common distributed
DataFrame abstraction across these different workflows.

**Given — cleanup.** Stop the producer with Ctrl+C in its terminal, then run the
cell below. It is also safe to rerun after an interrupted exercise. To begin
again after stopping Spark, restart the kernel and run from the top.


In [ ]:
import time
for active_query in spark.streams.active:
    active_query.stop()
spark.catalog.clearCache()
# Let any already scheduled background SQL jobs finish before closing the JVM.
deadline = time.monotonic() + 10
while spark.sparkContext.statusTracker().getActiveJobsIds() and time.monotonic() < deadline:
    time.sleep(0.1)
spark.stop()
print("All streaming queries and Spark stopped.")


References: [Spark DataFrames](https://spark.apache.org/docs/4.0.1/sql-programming-guide.html),
[ALS](https://spark.apache.org/docs/4.0.1/ml-collaborative-filtering.html),
[Structured Streaming](https://spark.apache.org/docs/4.0.1/streaming/apis-on-dataframes-and-datasets.html),
[MovieLens dataset attribution and terms](data/MOVIELENS_README.txt).
